# 🌏 NZ Earthquake Declustering — Multi-Method Comparison
### KDM vs HDBSCAN vs GMM vs Autoencoder

This notebook loads the output CSVs from all 4 methods and produces a unified comparison.

| Method | Type | Strength |
|---|---|---|
| **KDM (SOM)** | Unsupervised neural | Topology-preserving 2D map |
| **HDBSCAN** | Density-based | Variable-density, no assumptions |
| **GMM** | Probabilistic | Soft posteriors, interpretable components |
| **Autoencoder** | Deep learning | Learns non-linear feature interactions |

---
**Run this notebook AFTER completing all 4 method notebooks.**

## 📦 Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
def out(f): return str(OUTPUT_DIR / f)
print('✅ Imports done')

---
## 📂 Step 1 — Load All Method Results

In [ ]:
# Load all four declustered catalogues
files = {
    'KDM'         : 'outputs/NZ_declustered.csv',
    'HDBSCAN'     : 'outputs/NZ_HDBSCAN_declustered.csv',
    'GMM'         : 'outputs/NZ_GMM_declustered.csv',
    'Autoencoder' : 'outputs/NZ_Autoencoder_declustered.csv',
}

dfs = {}
for name, path in files.items():
    try:
        dfs[name] = pd.read_csv(path, low_memory=False)
        print(f'✅ {name:12s}: {len(dfs[name]):,} events loaded')
    except FileNotFoundError:
        print(f'⚠️  {name:12s}: file not found — run the method notebook first')

# Use the smallest common index (in case of minor differences after cleaning)
min_len = min(len(d) for d in dfs.values())
dfs     = {k: v.iloc[:min_len].reset_index(drop=True) for k,v in dfs.items()}
print(f'\nCommon length: {min_len:,} events')

---
## 📊 Step 2 — Crisis vs Non-Crisis Counts Comparison

In [ ]:
summary = []
for name, df in dfs.items():
    n_c  = (df['label']=='crisis').sum()
    n_nc = (df['label']=='non_crisis').sum()
    n    = len(df)
    summary.append({
        'Method'       : name,
        'Total'        : n,
        'Crisis'       : n_c,
        'Non_crisis'   : n_nc,
        'Pct_crisis'   : round(100*n_c/n, 1),
        'Pct_noncrisis': round(100*n_nc/n, 1)
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

fig, axes = plt.subplots(1,2, figsize=(14,5))

methods = [s['Method'] for s in summary]
pct_c   = [s['Pct_crisis'] for s in summary]
pct_nc  = [s['Pct_noncrisis'] for s in summary]
x       = np.arange(len(methods))
w       = 0.35

axes[0].bar(x - w/2, pct_c,  w, label='Crisis',     color='crimson',   alpha=0.8)
axes[0].bar(x + w/2, pct_nc, w, label='Non-crisis', color='steelblue', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(methods, fontsize=11)
axes[0].set_ylabel('% of catalogue')
axes[0].set_title('Crisis / Non-Crisis Proportion by Method', fontsize=12)
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
for i, (c, nc) in enumerate(zip(pct_c, pct_nc)):
    axes[0].text(i-w/2, c+0.5, f'{c}%', ha='center', fontsize=9)
    axes[0].text(i+w/2, nc+0.5, f'{nc}%', ha='center', fontsize=9)

# Absolute counts
n_c_abs  = [s['Crisis'] for s in summary]
n_nc_abs = [s['Non_crisis'] for s in summary]
axes[1].bar(x - w/2, n_c_abs,  w, label='Crisis',     color='crimson',   alpha=0.8)
axes[1].bar(x + w/2, n_nc_abs, w, label='Non-crisis', color='steelblue', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(methods, fontsize=11)
axes[1].set_ylabel('Number of events')
axes[1].set_title('Absolute Event Counts by Method', fontsize=12)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(out('COMPARE_event_counts.png'), dpi=150)
plt.show()

---
## 🤝 Step 3 — Agreement Matrix Between Methods

**Cohen's Kappa** measures how much two methods agree beyond chance.
- κ = 1.0 → perfect agreement
- κ = 0.0 → chance agreement
- κ < 0.0 → less than chance

In [ ]:
method_names = list(dfs.keys())
n_methods    = len(method_names)
kappa_matrix = np.zeros((n_methods, n_methods))
agree_matrix = np.zeros((n_methods, n_methods))

for i, m1 in enumerate(method_names):
    for j, m2 in enumerate(method_names):
        l1 = dfs[m1]['label'].values
        l2 = dfs[m2]['label'].values
        agree  = (l1 == l2).mean()
        kappa  = cohen_kappa_score(l1, l2)
        kappa_matrix[i,j] = round(kappa, 3)
        agree_matrix[i,j] = round(agree*100, 1)

fig, axes = plt.subplots(1,2, figsize=(14,5))

sns.heatmap(agree_matrix, annot=True, fmt='.1f', cmap='YlGn',
            xticklabels=method_names, yticklabels=method_names,
            ax=axes[0], vmin=50, vmax=100)
axes[0].set_title('Agreement (%) Between Methods', fontsize=12)

sns.heatmap(kappa_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
            xticklabels=method_names, yticklabels=method_names,
            ax=axes[1], vmin=-0.1, vmax=1.0)
axes[1].set_title("Cohen's Kappa Between Methods", fontsize=12)

plt.tight_layout()
plt.savefig(out('COMPARE_agreement_matrix.png'), dpi=150)
plt.show()

print('Agreement matrix (%):')
print(pd.DataFrame(agree_matrix, index=method_names, columns=method_names))

---
## 🗺️ Step 4 — Side-by-Side Spatial Maps (All 4 Methods)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for ax, (name, df) in zip(axes, dfs.items()):
    crisis     = df[df['label']=='crisis']
    non_crisis = df[df['label']=='non_crisis']
    ax.scatter(non_crisis['longitude'], non_crisis['latitude'],
               c='steelblue', s=0.3, alpha=0.3,
               label=f'Non-crisis ({len(non_crisis):,})')
    ax.scatter(crisis['longitude'], crisis['latitude'],
               c='crimson', s=0.3, alpha=0.3,
               label=f'Crisis ({len(crisis):,})')
    ax.set_title(f'{name} — NZ Declustering', fontsize=12)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.legend(markerscale=8, fontsize=9)

plt.suptitle('Side-by-Side Spatial Comparison — All Methods', fontsize=15)
plt.tight_layout()
plt.savefig(out('COMPARE_spatial_all_methods.png'), dpi=150)
plt.show()

---
## 📈 Step 5 — Cumulative Curves (All Methods on One Plot)

In [ ]:
time_col = 'time' if 'time' in list(dfs.values())[0].columns else 'Year'

colors_crisis = ['crimson','darkorange','purple','darkred']
colors_bg     = ['steelblue','teal','navy','royalblue']

fig, ax = plt.subplots(figsize=(16, 6))

for i, (name, df) in enumerate(dfs.items()):
    df_s = df.sort_values(time_col)
    N    = len(df_s)
    c_s  = df_s[df_s['label']=='crisis']
    nc_s = df_s[df_s['label']=='non_crisis']

    ax.plot(c_s[time_col],  np.arange(len(c_s))/N,
            color=colors_crisis[i], linestyle=':', lw=1.5,
            label=f'{name} — Crisis ({len(c_s):,})')
    ax.plot(nc_s[time_col], np.arange(len(nc_s))/N,
            color=colors_bg[i], linestyle='-', lw=1.5,
            label=f'{name} — Non-crisis ({len(nc_s):,})')

# Full catalogue
df_ref = list(dfs.values())[0].sort_values(time_col)
ax.plot(df_ref[time_col], np.arange(len(df_ref))/len(df_ref),
        'k--', lw=1, alpha=0.4, label='Full catalogue')

ax.set_xlabel('Time (decimal years)', fontsize=12)
ax.set_ylabel('Proportion of events', fontsize=12)
ax.set_title('Cumulative Curves — All Methods Compared', fontsize=14)
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out('COMPARE_cumulative_all.png'), dpi=150)
plt.show()

---
## 🔬 Step 6 — Consensus Labelling

Events where **3 or 4 methods agree** get a 'high-confidence consensus' label.
This is the most reliable declustered catalogue.

In [ ]:
# Build consensus
crisis_votes = sum(
    (dfs[m]['label']=='crisis').astype(int) for m in method_names
)

df_consensus = list(dfs.values())[0][['latitude','longitude','magnitude',time_col]].copy()
df_consensus['crisis_votes']     = crisis_votes
df_consensus['consensus_label']  = np.where(crisis_votes >= 3, 'crisis', 'non_crisis')
df_consensus['consensus_conf']   = (crisis_votes / len(method_names) - 0.5).abs() / 0.5

# High confidence = unanimous (4/4 or 0/4)
df_consensus['high_confidence'] = crisis_votes.isin([0, len(method_names)])

n_c  = (df_consensus['consensus_label']=='crisis').sum()
n_nc = (df_consensus['consensus_label']=='non_crisis').sum()
n_hc = df_consensus['high_confidence'].sum()

print(f'\n══ CONSENSUS Summary ══════════════════════════')
print(f'  Total          : {len(df_consensus):,}')
print(f'  Crisis (≥3/4)  : {n_c:,}  ({100*n_c/len(df_consensus):.1f}%)')
print(f'  Non-crisis     : {n_nc:,}  ({100*n_nc/len(df_consensus):.1f}%)')
print(f'  Unanimous (4/4): {n_hc:,}  ({100*n_hc/len(df_consensus):.1f}%)')
print(f'══════════════════════════════════════════════')

# Vote distribution
fig, axes = plt.subplots(1,2, figsize=(14,5))

vote_counts = crisis_votes.value_counts().sort_index()
axes[0].bar(vote_counts.index, vote_counts.values,
            color=['steelblue','lightblue','lightsalmon','salmon','crimson'],
            edgecolor='white', alpha=0.9)
axes[0].set_xticks(range(len(method_names)+1))
axes[0].set_xticklabels([f'{i}/{len(method_names)} methods' for i in range(len(method_names)+1)])
axes[0].set_ylabel('Number of events')
axes[0].set_title('Crisis Vote Distribution (how many methods agree)', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

sc = axes[1].scatter(df_consensus['longitude'], df_consensus['latitude'],
                     c=df_consensus['crisis_votes'], cmap='RdYlGn_r',
                     s=0.5, alpha=0.4, vmin=0, vmax=len(method_names))
plt.colorbar(sc, ax=axes[1], label='Number of methods calling crisis')
axes[1].set_title('Consensus Map — Crisis Vote Count', fontsize=11)
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.savefig(out('COMPARE_consensus.png'), dpi=150)
plt.show()

df_consensus.to_csv(out('NZ_consensus_declustered.csv'), index=False)
print('✅ Saved → NZ_consensus_declustered.csv')

---
## 📋 Step 7 — Final Comparison Table

In [ ]:
print('\n══ FULL COMPARISON TABLE ═══════════════════════════════════════')
print(f'{"Method":<14} {"Crisis%":>8} {"NonCrisis%":>11} {"Agree_KDM%":>12} {"Kappa_KDM":>10}')
print('-'*60)

kdm_labels = dfs['KDM']['label'].values
for name, df in dfs.items():
    n     = len(df)
    pct_c = 100*(df['label']=='crisis').sum()/n
    pct_nc= 100*(df['label']=='non_crisis').sum()/n
    if name != 'KDM':
        agree = 100*(df['label'].values == kdm_labels).mean()
        kappa = cohen_kappa_score(df['label'].values, kdm_labels)
    else:
        agree, kappa = 100.0, 1.0
    print(f'{name:<14} {pct_c:>7.1f}%  {pct_nc:>10.1f}%  {agree:>10.1f}%  {kappa:>10.3f}')

print('═'*60)
print('\n✅ All comparison plots saved to outputs/ folder')